# 🧾 Notebook 2: Merkle Inclusion Proofs

**The scenario:** a *light client* knows only the 32-byte **root hash** of a huge dataset (millions of leaves). Someone claims "hey, `alice=100` is in there — trust me." How can the light client verify this claim *without* downloading the whole dataset?

Answer: a **Merkle inclusion proof**. The prover sends:
- the leaf value,
- its position (leaf index),
- and only `log₂(N)` sibling hashes along the path from the leaf to the root.

The verifier re-computes the root from those. If it matches the known root, the leaf *must* have been part of the tree — otherwise the prover would have had to find a hash collision.

This is how **Bitcoin SPV wallets**, **Certificate Transparency** audits, **IPFS**, and **blockchain light clients** work.

## Learning objectives
- Compare three ways to prove membership: send everything (🟥), send all hashes (🟨), send a Merkle proof (🟩).
- Implement proof generation and verification.
- See that proof size is `O(log N)` — ~20 hashes for a million-leaf tree.

In [ ]:
import hashlib
import math

def H_leaf(data: bytes) -> bytes:
    return hashlib.sha256(b'\x00' + data).digest()

def H_node(left: bytes, right: bytes) -> bytes:
    return hashlib.sha256(b'\x01' + left + right).digest()

def build_merkle(leaves: list[bytes]) -> list[list[bytes]]:
    if not leaves:
        return [[H_leaf(b'')]]
    level = [H_leaf(x) for x in leaves]
    levels = [level]
    while len(level) > 1:
        if len(level) % 2 == 1:
            level = level + [level[-1]]
        level = [H_node(level[i], level[i + 1]) for i in range(0, len(level), 2)]
        levels.append(level)
    return levels

# Dataset of 1024 account balances. Real trees have millions of leaves.
accounts = [f'user{i:04d}=${i * 7}'.encode() for i in range(1024)]
tree = build_merkle(accounts)
root = tree[-1][0]
print(f'{len(accounts)} leaves, root = {root.hex()[:16]}...')

## 🟥 Bad: "just send me everything"

To convince the light client, the server sends the **entire dataset** and the client re-hashes it. This works, but it defeats the point of a light client.

In [ ]:
def bad_prove(accounts, index):
    payload = b'|'.join(accounts)  # send everything
    return {'payload': payload, 'index': index}, len(payload)

def bad_verify(proof, expected_root):
    items = proof['payload'].split(b'|')
    return build_merkle(items)[-1][0] == expected_root

proof, size = bad_prove(accounts, 3)
bad_size = size
print(f'BAD  : proof size = {size} bytes, verified = {bad_verify(proof, root)}')
assert bad_verify(proof, root)

## 🟨 OK: "send every leaf hash"

Better: send the 32-byte hash of every leaf. Smaller than full data but still `O(N)` — a million leaves = 32 MB.

In [ ]:
def ok_prove(accounts, index):
    leaf_hashes = [H_leaf(x) for x in accounts]
    size = 32 * len(leaf_hashes)
    return {'leaf_hashes': leaf_hashes, 'index': index, 'leaf': accounts[index]}, size

def ok_verify(proof, expected_root):
    hashes = list(proof['leaf_hashes'])
    if H_leaf(proof['leaf']) != hashes[proof['index']]:
        return False
    while len(hashes) > 1:
        if len(hashes) % 2 == 1:
            hashes.append(hashes[-1])
        hashes = [H_node(hashes[i], hashes[i + 1]) for i in range(0, len(hashes), 2)]
    return hashes[0] == expected_root

proof, size = ok_prove(accounts, 3)
ok_size = size
print(f'OK   : proof size = {size} bytes, verified = {ok_verify(proof, root)}')
assert ok_verify(proof, root)
assert ok_size == 32 * len(accounts)      # O(N): one hash per leaf

## 🟩 Best: a **Merkle proof** — only siblings along the path

To re-compute the root from a single leaf, you need exactly the **sibling hash at each level**. That's `⌈log₂ N⌉` hashes — about **20 hashes for a million leaves**. You also need to know whether each sibling sits to the *left* or to the *right* so you combine them in the right order.

```
       root
      /    \
    H01    H23      ← need H23 (the sibling of H01)
   /   \   /  \
  H0   H1 H2  H3    ← need H1 (the sibling of H0)
  ^
  our leaf (index 0)
```

For leaf at index 0 we send `[(H1, 'right'), (H23, 'right')]` — 2 hashes for 4 leaves.

In [ ]:
def merkle_proof(tree, index):
    """Return list of (sibling_hash, side) pairs from leaf up to root.
    side='left' means sibling is on the left, so combine as H_node(sibling, current).
    side='right' means sibling is on the right, so combine as H_node(current, sibling).
    """
    proof = []
    for level in tree[:-1]:  # all but root
        # duplicate last if odd, to mirror how the tree was built
        nodes = level + [level[-1]] if len(level) % 2 == 1 else level
        sibling_idx = index ^ 1  # flip the lowest bit: 0↔1, 2↔3, ...
        side = 'left' if sibling_idx < index else 'right'
        proof.append((nodes[sibling_idx], side))
        index //= 2
    return proof

def verify_proof(leaf: bytes, index: int, proof, expected_root: bytes) -> bool:
    h = H_leaf(leaf)
    for sibling, side in proof:
        h = H_node(sibling, h) if side == 'left' else H_node(h, sibling)
    return h == expected_root

# Prove that accounts[3] is in the tree.
idx = 3
p = merkle_proof(tree, idx)
proof_size = 32 * len(p) + 4  # hashes + 4 bytes for the index
print(f'BEST : proof size = {proof_size} bytes ({len(p)} sibling hashes)')
print(f'       verified   = {verify_proof(accounts[idx], idx, p, root)}')

# The proof is exactly one sibling per level: ceil(log2 N), no more.
assert len(p) == math.ceil(math.log2(len(accounts))) == 10
assert verify_proof(accounts[idx], idx, p, root)
# And it must work for EVERY leaf, not just the one we picked.
assert all(verify_proof(accounts[i], i, merkle_proof(tree, i), root)
           for i in range(len(accounts)))
print(f'\n✔ all {len(accounts)} leaves verify; {proof_size} bytes vs {ok_size:,} (OK) '
      f'vs {bad_size:,} (BAD)')
print(f'  that is {ok_size / proof_size:.0f}x smaller than sending every leaf hash')

## 🔬 Tamper test — a forged leaf fails

If someone lies about the leaf value, re-computing up the path produces a different root and verification fails. No way to forge without finding a SHA-256 collision.

In [ ]:
fake_leaf = b'user003=$999999'  # attacker's claim
print('honest leaf verifies? ', verify_proof(accounts[idx], idx, p, root))
print('forged leaf verifies? ', verify_proof(fake_leaf, idx, p, root))

assert verify_proof(accounts[idx], idx, p, root) is True
assert verify_proof(fake_leaf, idx, p, root) is False

# Three more ways a prover might try to cheat, all of which must fail:
# 1. A real leaf presented at the wrong index (its path no longer matches).
assert verify_proof(accounts[idx], idx, merkle_proof(tree, idx + 1), root) is False
# 2. A tampered sibling hash anywhere along the path.
for level in range(len(p)):
    bad = list(p)
    sib, side = bad[level]
    bad[level] = (bytes(sib[0] ^ 0x01) + sib[1:], side)
    assert verify_proof(accounts[idx], idx, bad, root) is False, level
# 3. Passing an INTERNAL node off as a leaf — the attack domain separation blocks.
#    tree[1][0] is a real internal hash, but H_leaf of it can never reach the root.
internal_node = tree[1][0]
assert verify_proof(internal_node, 0, merkle_proof(tree, 0), root) is False
print('\n✔ forged leaf, wrong index, tampered sibling, and internal-node-as-leaf all rejected')

### Does it still work when `N` is not a power of two?

Our 1024-leaf tree is a perfectly balanced binary tree, which is the easy case. Real datasets
are never a power of two, so the bottom level gets **padded** by duplicating the last node —
and the proof generator has to mirror that padding exactly, or the last few leaves produce
proofs that don't verify. This is a classic off-by-one, so let's check it across many sizes
rather than trusting one lucky example.

In [ ]:
for n in [1, 2, 3, 5, 7, 100, 511, 1000]:
    items = [f'item{i}'.encode() for i in range(n)]
    t = build_merkle(items)
    r = t[-1][0]
    for i in range(n):
        assert verify_proof(items[i], i, merkle_proof(t, i), r), (n, i)
    print(f'N={n:>4}: all {n} inclusion proofs verify '
          f'({len(merkle_proof(t, 0))} siblings, ceil(log2 N) = {math.ceil(math.log2(n)) if n > 1 else 0})')

print('\n✔ padding handled correctly — the last leaf of an odd level is the one that breaks '
      'naive implementations')

## 📈 Proof size vs dataset size

The real payoff: proofs grow **logarithmically**. Compare payload sizes for a range of tree sizes.

In [ ]:
import math
import matplotlib.pyplot as plt

sizes = [2**k for k in range(1, 21)]  # up to ~1M leaves
bad  = [32 * n for n in sizes]                  # sending all leaf hashes
best = [32 * math.ceil(math.log2(n)) for n in sizes]  # Merkle proof

plt.figure(figsize=(7, 4))
plt.plot(sizes, bad, label='All leaf hashes (O(N))', marker='o')
plt.plot(sizes, best, label='Merkle proof (O(log N))', marker='s')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Number of leaves'); plt.ylabel('Bytes sent')
plt.title('Proof size grows logarithmically with Merkle')
plt.grid(True, which='both', alpha=0.3); plt.legend()
plt.tight_layout(); plt.show()
print(f'1M leaves → Merkle proof ≈ {32 * math.ceil(math.log2(10**6))} bytes (~{math.ceil(math.log2(10**6))} hashes)')

## ✅ Recap
- A Merkle **inclusion proof** is `O(log N)` sibling hashes + the leaf's position.
- The verifier only needs the **root hash** (one 32-byte value).
- Real-world uses: Bitcoin SPV wallets, Certificate Transparency audits, IPFS block fetches, blockchain light clients.
- Forging a proof would require finding a SHA-256 collision — currently infeasible.